# Topic Co-Authorship Evolution and IDR Analysis

This notebook analyzes topic frequency over time, topic-specific co-authorship network structure, researcher centrality/IDR patterns, and community-level topic diversity. Use it to compare the earlier period (2016-2020) with the later period (2021-2024), inspect a selected topic network, and review people/community metrics.

## Setup
Run the following cells first to ensure the notebook is using the project virtual environment and required helper modules.

In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print('PROJECT_ROOT:', PROJECT_ROOT)
print('Python:', sys.executable)

PROJECT_ROOT: /Users/dharani/Desktop/PSI
Python: /Users/dharani/Desktop/PSI/env/bin/python


In [2]:
%pip install -r '../requirements.txt'


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
from scripts.idr_analysis import build_idr_outputs, load_base_frames
from scripts.topic_network_analysis import (
    available_topics,
    build_topic_report,
    load_filtered_papers,
    summary_to_series,
    split_semicolon_values,
)

DATA_PATH = PROJECT_ROOT / 'data' / 'papers_filtered.csv'
CENTRALITY_PATH = PROJECT_ROOT / 'data' / 'centrality.csv'

def period_label(year: int) -> str:
    if year <= 2020:
        return '2016-2020'
    return '2021-2024'

def compute_graph_diameter(authors_df: pd.DataFrame, edges_df: pd.DataFrame) -> int:
    if authors_df.empty or edges_df.empty:
        return 0

    adjacency = {author: set() for author in authors_df['author']}
    for _, row in edges_df.iterrows():
        adjacency[row['source']].add(row['target'])
        adjacency[row['target']].add(row['source'])

    def bfs_distance(start: str) -> int:
        seen = {start}
        queue = [(start, 0)]
        max_dist = 0
        while queue:
            node, dist = queue.pop(0)
            max_dist = max(max_dist, dist)
            for neighbor in adjacency[node]:
                if neighbor not in seen:
                    seen.add(neighbor)
                    queue.append((neighbor, dist + 1))
        return max_dist

    largest_cc = []
    visited = set()
    for node in adjacency:
        if node in visited:
            continue
        component = {node}
        queue = [node]
        visited.add(node)
        while queue:
            current = queue.pop(0)
            for neighbor in adjacency[current]:
                if neighbor not in visited:
                    visited.add(neighbor)
                    component.add(neighbor)
                    queue.append(neighbor)
        if len(component) > len(largest_cc):
            largest_cc = list(component)

    if not largest_cc:
        return 0

    return max(bfs_distance(node) for node in largest_cc)

## Data Overview and Topic Evolution
Load the filtered paper dataset, create periods for the two eras, and compare topic frequency between 2016-2020 and 2021-2024.

In [4]:
papers_df = pd.read_csv(DATA_PATH)
papers_df['topic_list'] = papers_df['topics'].apply(split_semicolon_values)
papers_df['period'] = papers_df['year'].apply(period_label)

print('Papers loaded:', len(papers_df))
print('Unique periods:', papers_df['period'].unique().tolist())

print('Available topics:')
available = available_topics(papers_df)
print(len(available), 'topics found')
print(available[:20])

Papers loaded: 2743
Unique periods: ['2016-2020', '2021-2024']
Available topics:
68 topics found
['Ag economics', 'Agtech', 'artificial intelligence', 'automation', 'bioremediation', 'biotic stress', 'breeding', 'carbon capture', 'community', 'computer vision', 'conservation', 'CPS', 'CRISPR', 'data analytics', 'data science', 'decision support', 'digital ag', 'digital twin', 'digital twins', 'disease control']


In [6]:
topic_counts = (
    papers_df.explode('topic_list')
    .dropna(subset=['topic_list'])
    .assign(topic=lambda df: df['topic_list'].astype(str).str.strip().str.lower())
    .query('topic != ""')
    .groupby(['period', 'topic'])['DOI']
    .count()
    .reset_index(name='paper_count')
)

topic_counts['period_share'] = topic_counts.groupby('period')['paper_count'].transform(lambda x: x / x.sum())

topic_pivot = topic_counts.pivot(index='topic', columns='period', values='paper_count').fillna(0).astype(int)
topic_pivot = topic_pivot.sort_values(by=['2016-2020', '2021-2024'], ascending=False).head(30)
topic_pivot


period,2016-2020,2021-2024
topic,,
soil,1282,1257
microbes,1278,1243
genomics,1277,1271
ecology,1276,1240
science communication,1272,1241
fungi,1269,1219
sensing,1268,1233
modeling,1266,1248
genetics,1266,1246


## Selected Topic Co-Authorship Network
Build the topic network for a selected topic and inspect network density, clustering, and author centrality. Change `SELECTED_TOPIC` to analyze a different topic.

In [7]:
SELECTED_TOPIC = 'carbon capture'
report = build_topic_report(SELECTED_TOPIC, DATA_PATH)
topic_summary = summary_to_series(report['summary'])
topic_summary

papers_in_topic                                                            352
authors_in_topic_network                                                  1673
nc_authors_in_topic_network                                                243
external_authors_in_topic_network                                         1430
coauthor_edges                                                           14583
density                                                               0.010427
components                                                                  57
largest_component_size                                                     624
largest_component_share                                               0.372983
average_local_clustering                                              0.917952
nc_nc_edges                                                                410
nc_external_edges                                                         1775
external_external_edges                             

In [8]:
print('Network diameter (largest connected component):', compute_graph_diameter(report['topic_authors_df'], report['topic_edges_df']))

report['top_authors_df'][[
    'author',
    'nc_state',
    'topic_paper_count',
    'topic_degree',
    'topic_weighted_degree',
    'external_neighbors',
    'external_neighbor_share',
    'local_clustering',
]].head(10)

Network diameter (largest connected component): 18


,author,nc_state,topic_paper_count,topic_degree,topic_weighted_degree,external_neighbors,external_neighbor_share,local_clustering
0,"Jones, Jacob L.",False,17,97,127,82,0.845361,0.099012
1,"Meentemeyer, Ross K.",False,16,60,78,44,0.733333,0.110169
2,"Salvi, Deepti",True,14,24,34,21,0.875000,0.144928
3,"Theis, Thomas",True,13,41,89,34,0.829268,0.240244
4,"Stapelmann, Katharina",True,12,51,70,48,0.941176,0.141961
5,"Daniele, Michael A.",False,12,52,65,31,0.596154,0.138763
6,"Isik, Fikret",True,11,41,46,39,0.951220,0.192683
7,"Chekmenev, Eduard Y.",False,10,38,74,30,0.789474,0.263158
8,"Jordan, David L.",False,9,42,56,33,0.785714,0.180023
9,"Wei, Qingshan",True,9,43,51,21,0.488372,0.181617


## Researcher Centrality and IDR
Use the IDR pipeline to compare researcher interdisciplinarity and connect it with centrality metadata.

In [9]:
papers_df, centrality_df = load_base_frames(DATA_PATH, CENTRALITY_PATH)
idr_outputs = build_idr_outputs(DATA_PATH, CENTRALITY_PATH)
people_idr_df = idr_outputs['people_idr_df']
community_idr_df = idr_outputs['community_idr_df']

print('People IDR rows:', len(people_idr_df))
print('Community IDR rows:', len(community_idr_df))

people_idr_df[['author', 'nc_state', 'paper_count', 'rao_stirling', 'normalized_shannon', 'num_topics']].head(15)

People IDR rows: 10032
Community IDR rows: 47


,author,nc_state,paper_count,rao_stirling,normalized_shannon,num_topics
0,"Choe, Kisurb",False,1,0.486769,1.0,4
1,"Thomas, Anna",False,1,0.486769,1.0,4
2,"Brawner, Jeremy",False,1,0.439459,1.0,7
3,"Hudson, Owen",False,1,0.439459,1.0,7
4,"Messina, Charlie",False,1,0.439459,1.0,7
5,"Resende Jr, Marcio F. R.",False,1,0.439459,1.0,7
6,"Crawford, Kerri M.",False,1,0.437158,1.0,9
7,"Bello-Bello, Elohim",False,1,0.431709,1.0,11
8,"Olalde-Portugal, Victor",False,1,0.431709,1.0,11
9,"Rosario Ramirez-Flores, M.",False,1,0.431709,1.0,11


## Community Topic Diversity
Review community-level topic diversity metrics and compare community size with Rao-Stirling and entropy measures.

In [10]:
community_idr_df[['community', 'author_count', 'paper_count', 'num_topics', 'rao_stirling', 'normalized_shannon']].head(20)

,community,author_count,paper_count,num_topics,rao_stirling,normalized_shannon
0,6653,133,56,67,0.334844,0.978762
1,4173,243,39,67,0.321712,0.952202
2,3931,93,28,67,0.296609,0.957075
3,6160,84,31,68,0.288487,0.950109
4,8793,18,5,67,0.281148,0.950245
5,3828,436,182,68,0.266027,0.937219
6,2210,275,76,67,0.265214,0.935126
7,2479,259,113,68,0.254912,0.932801
8,8660,221,79,67,0.251304,0.932735
9,5649,220,111,68,0.242850,0.924383
